# Weather Dataset Loading & Preprocessing Pipeline

This notebook demonstrates a professional workflow for:

- Extracting ERA5 weather datasets
- Loading NetCDF climate files
- Merging temperature and precipitation datasets
- Selecting a single geographic location
- Performing feature engineering
- Preparing the dataset for machine learning and time-series forecasting

---

## Dataset Source
- ERA5 Reanalysis Dataset
- Copernicus Climate Data Store (CDS)

## Target Region
- Azamgarh, Uttar Pradesh, India


## Import Required Libraries

In [1]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import os

## Configuration

In [2]:
# Quarter folders
# QUARTERS = ["Q1", "Q2", "Q3", "Q4"]

# Monthly folders
MONTHS = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]
# Target coordinate (Azamgarh)
TARGET_LATITUDE = 26.00
TARGET_LONGITUDE = 83.00

# Dataset directory
BASE_DIR = os.path.dirname(os.getcwd())
DATASET_DIR = Path(BASE_DIR) / "data"

## Extract ZIP Files

In [3]:
for month in MONTHS:

    zip_path = DATASET_DIR / "archives" / f"azamgarh_weather_{month}_2025.zip"
    extract_path = DATASET_DIR / "raw" / month

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    print(f"{month} extracted successfully.")

JAN extracted successfully.
FEB extracted successfully.
MAR extracted successfully.
APR extracted successfully.
MAY extracted successfully.
JUN extracted successfully.
JUL extracted successfully.
AUG extracted successfully.
SEP extracted successfully.
OCT extracted successfully.
NOV extracted successfully.
DEC extracted successfully.


## Load Instantaneous Weather Variables

This dataset contains:
- Temperature
- Pressure
- Wind
- Cloud Cover


In [3]:
instant_dfs = []

for month in MONTHS:

    file_path = (
        DATASET_DIR /
        "raw" /
        month /
        "data_stream-oper_stepType-instant.nc"
    )

    with xr.open_dataset(
        file_path,
        engine="netcdf4"
    ) as ds:

        df = ds.to_dataframe().reset_index()
    
    instant_dfs.append(df)

# Combine all months
instant_df = pd.concat(
    instant_dfs,
    ignore_index=True
)

print(instant_df.shape)
instant_df.head()

(175200, 14)


,valid_time,latitude,longitude,t2m,d2m,sp,u10,v10,tcc,lcc,mcc,hcc,number,expver
0,2025-01-01,26.25,82.50,285.763702,283.003204,100675.953125,2.229599,-0.069473,0.332184,0.332184,0.0,0.0,0,0001
1,2025-01-01,26.25,82.75,285.443390,283.120392,100806.953125,2.273544,0.044785,0.250854,0.250854,0.0,0.0,0,0001
2,2025-01-01,26.25,83.00,284.988312,283.137970,100756.953125,2.308701,0.199081,0.252563,0.252563,0.0,0.0,0,0001
3,2025-01-01,26.25,83.25,284.718781,283.155548,100808.953125,2.416122,0.395370,0.190643,0.190643,0.0,0.0,0,0001
4,2025-01-01,26.25,83.50,284.703156,283.309845,100886.953125,2.620224,0.531113,0.118958,0.118958,0.0,0.0,0,0001


## Load Precipitation Dataset

In [4]:
precipitation_dfs = []

for month in MONTHS:

    file_path = (
        DATASET_DIR /
        "raw" /
        month /
        "data_stream-oper_stepType-accum.nc"
    )

    with xr.open_dataset(
        file_path,
        engine="netcdf4"
    ) as ds:

        df = ds.to_dataframe().reset_index()

    precipitation_dfs.append(df)

# Combine all quarters
precipitation_df = pd.concat(
    precipitation_dfs,
    ignore_index=True
)

print(precipitation_df.shape)
precipitation_df.head()

(175200, 6)


,valid_time,latitude,longitude,tp,number,expver
0,2025-01-01,26.25,82.50,0.0,0,0001
1,2025-01-01,26.25,82.75,0.0,0,0001
2,2025-01-01,26.25,83.00,0.0,0,0001
3,2025-01-01,26.25,83.25,0.0,0,0001
4,2025-01-01,26.25,83.50,0.0,0,0001


## Extract Single Location Time-Series

In [5]:
single_temp_df = instant_df[
    (instant_df['latitude'] == TARGET_LATITUDE) &
    (instant_df['longitude'] == TARGET_LONGITUDE)
].reset_index(drop=True)

single_precipitation_df = precipitation_df[
    (precipitation_df['latitude'] == TARGET_LATITUDE) &
    (precipitation_df['longitude'] == TARGET_LONGITUDE)
].reset_index(drop=True)

print(single_temp_df.shape)
print(single_precipitation_df.shape)

(8760, 14)
(8760, 6)


## Merge Temperature & Precipitation Data

In [6]:
single_temp_df["tp"] = (
    single_precipitation_df["tp"].values
)

weather_df = single_temp_df.copy()

In [7]:
weather_df.head()

,valid_time,latitude,longitude,t2m,d2m,sp,u10,v10,tcc,lcc,mcc,hcc,number,expver,tp
0,2025-01-01 00:00:00,26.0,83.0,285.455109,283.341095,100862.953125,2.012802,0.118027,0.303497,0.303497,0.0,0.0,0,0001,0.000002
1,2025-01-01 01:00:00,26.0,83.0,285.764496,283.411926,100916.000000,2.368378,0.273438,0.468201,0.468201,0.0,0.0,0,0001,0.000006
2,2025-01-01 02:00:00,26.0,83.0,285.656281,283.078522,101011.937500,2.863358,-0.085052,0.496887,0.496887,0.0,0.0,0,0001,0.000000
3,2025-01-01 03:00:00,26.0,83.0,286.015350,282.391754,101072.265625,2.798767,-0.625854,0.428497,0.428497,0.0,0.0,0,0001,0.000000
4,2025-01-01 04:00:00,26.0,83.0,285.377502,280.956665,101105.250000,3.277405,-0.995514,0.381317,0.381317,0.0,0.0,0,0001,0.000000


In [8]:
# =========================
# OUTPUT CONFIGURATION
# =========================

OUTPUT_DIR = DATASET_DIR / "processed"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "azamgarh_weather_raw.csv"
)


# =========================
# SAVE DATASET
# =========================

weather_df.to_csv(
    OUTPUT_FILE,
    index=False
)


# =========================
# CONFIRMATION
# =========================

print("\nDataset saved successfully.")

print(f"\nLocation: {OUTPUT_FILE}")

print(f"\nTotal Rows: {weather_df.shape[0]}")

print(f"Total Columns: {weather_df.shape[1]}")


Dataset saved successfully.

Location: c:\Users\DELL\OneDrive\Desktop\weather-forecasting-system\data\processed\azamgarh_weather_raw.csv

Total Rows: 8760
Total Columns: 15


## Rename Columns

In [10]:
weather_df = weather_df.rename(columns={
    "t2m": "temperature",
    
    "sp": "surface_pressure",
    "u10": "wind_u",
    "v10": "wind_v",
    "tcc": "total_cloud_cover",
    "lcc": "low_cloud_cover",
    "mcc": "medium_cloud_cover",
    "hcc": "high_cloud_cover",
    "tp": "precipitation"
})

weather_df.head()

,valid_time,latitude,longitude,temperature,d2m,surface_pressure,wind_u,wind_v,total_cloud_cover,low_cloud_cover,medium_cloud_cover,high_cloud_cover,number,expver,precipitation
0,2025-01-01 00:00:00,26.0,83.0,285.455109,283.341095,100862.953125,2.012802,0.118027,0.303497,0.303497,0.0,0.0,0,0001,0.000002
1,2025-01-01 01:00:00,26.0,83.0,285.764496,283.411926,100916.000000,2.368378,0.273438,0.468201,0.468201,0.0,0.0,0,0001,0.000006
2,2025-01-01 02:00:00,26.0,83.0,285.656281,283.078522,101011.937500,2.863358,-0.085052,0.496887,0.496887,0.0,0.0,0,0001,0.000000
3,2025-01-01 03:00:00,26.0,83.0,286.015350,282.391754,101072.265625,2.798767,-0.625854,0.428497,0.428497,0.0,0.0,0,0001,0.000000
4,2025-01-01 04:00:00,26.0,83.0,285.377502,280.956665,101105.250000,3.277405,-0.995514,0.381317,0.381317,0.0,0.0,0,0001,0.000000


## Feature Engineering

In [28]:
# Temperature: Kelvin → Celsius
weather_df["temperature"] = (
    weather_df["temperature"] - 273.15
)

weather_df['d2m'] = (
    weather_df['d2m'] - 273.15
)

# Relative Humidity (%)
weather_df["humidity"] = 100 * (
    np.exp(
        (17.625 * weather_df['d2m']) /
        (243.04 + weather_df['d2m'])
    )
    /
    np.exp(
        (17.625 * weather_df["temperature"]) /
        (243.04 + weather_df["temperature"])
    )
)

# Optional clipping
weather_df["humidity"] = weather_df["humidity"].clip(0, 100)

# Pressure: Pa → hPa
weather_df["surface_pressure"] = (
    weather_df["surface_pressure"] / 100
)

# Precipitation: meters → millimeters
weather_df["precipitation"] = (
    weather_df["precipitation"] * 1000
)

# Cloud Cover: fraction → percentage
cloud_columns = [
    "total_cloud_cover",
    "low_cloud_cover",
    "medium_cloud_cover",
    "high_cloud_cover"
]

weather_df[cloud_columns] = (
    weather_df[cloud_columns] * 100
)

# Wind Speed
weather_df["wind_speed"] = np.sqrt(
    weather_df["wind_u"]**2 +
    weather_df["wind_v"]**2
)

weather_df.head()

,valid_time,latitude,longitude,temperature,d2m,surface_pressure,wind_u,wind_v,total_cloud_cover,low_cloud_cover,medium_cloud_cover,high_cloud_cover,number,expver,precipitation,humidity,wind_speed
0,2025-01-01 00:00:00,26.0,83.0,12.305115,10.191101,1008.629517,2.012802,0.118027,30.349731,30.349731,0.0,0.0,0,0001,0.001907,86.931839,2.016260
1,2025-01-01 01:00:00,26.0,83.0,12.614502,10.261932,1009.159973,2.368378,0.273438,46.820068,46.820068,0.0,0.0,0,0001,0.006199,85.588684,2.384110
2,2025-01-01 02:00:00,26.0,83.0,12.506287,9.928528,1010.119385,2.863358,-0.085052,49.688721,49.688721,0.0,0.0,0,0001,0.000000,84.298172,2.864620
3,2025-01-01 03:00:00,26.0,83.0,12.865356,9.241760,1010.722656,2.798767,-0.625854,42.849731,42.849731,0.0,0.0,0,0001,0.000000,78.629288,2.867890
4,2025-01-01 04:00:00,26.0,83.0,12.227509,7.806671,1011.052490,3.277405,-0.995514,38.131714,38.131714,0.0,0.0,0,0001,0.000000,74.398216,3.425264


## Time-Based Features

In [29]:
weather_df["valid_time"] = pd.to_datetime(
    weather_df["valid_time"]
)

weather_df["hour"] = (
    weather_df["valid_time"].dt.hour
)

weather_df["day"] = (
    weather_df["valid_time"].dt.day
)

weather_df["month"] = (
    weather_df["valid_time"].dt.month
)

weather_df["day_of_week"] = (
    weather_df["valid_time"].dt.dayofweek
)

weather_df.head()

,valid_time,latitude,longitude,temperature,d2m,surface_pressure,wind_u,wind_v,total_cloud_cover,low_cloud_cover,...,high_cloud_cover,number,expver,precipitation,humidity,wind_speed,hour,day,month,day_of_week
0,2025-01-01 00:00:00,26.0,83.0,12.305115,10.191101,1008.629517,2.012802,0.118027,30.349731,30.349731,...,0.0,0,0001,0.001907,86.931839,2.016260,0,1,1,2
1,2025-01-01 01:00:00,26.0,83.0,12.614502,10.261932,1009.159973,2.368378,0.273438,46.820068,46.820068,...,0.0,0,0001,0.006199,85.588684,2.384110,1,1,1,2
2,2025-01-01 02:00:00,26.0,83.0,12.506287,9.928528,1010.119385,2.863358,-0.085052,49.688721,49.688721,...,0.0,0,0001,0.000000,84.298172,2.864620,2,1,1,2
3,2025-01-01 03:00:00,26.0,83.0,12.865356,9.241760,1010.722656,2.798767,-0.625854,42.849731,42.849731,...,0.0,0,0001,0.000000,78.629288,2.867890,3,1,1,2
4,2025-01-01 04:00:00,26.0,83.0,12.227509,7.806671,1011.052490,3.277405,-0.995514,38.131714,38.131714,...,0.0,0,0001,0.000000,74.398216,3.425264,4,1,1,2


## Rolling Features

In [30]:
weather_df["temp_rolling_6"] = (
    weather_df["temperature"]
    .rolling(6)
    .mean()
)

weather_df["temp_lag_1"] = (
    weather_df["temperature"]
    .shift(1)
)

weather_df["temp_lag_24"] = (
    weather_df["temperature"]
    .shift(24)
)

weather_df.head()

,valid_time,latitude,longitude,temperature,d2m,surface_pressure,wind_u,wind_v,total_cloud_cover,low_cloud_cover,...,precipitation,humidity,wind_speed,hour,day,month,day_of_week,temp_rolling_6,temp_lag_1,temp_lag_24
0,2025-01-01 00:00:00,26.0,83.0,12.305115,10.191101,1008.629517,2.012802,0.118027,30.349731,30.349731,...,0.001907,86.931839,2.016260,0,1,1,2,NaN,NaN,NaN
1,2025-01-01 01:00:00,26.0,83.0,12.614502,10.261932,1009.159973,2.368378,0.273438,46.820068,46.820068,...,0.006199,85.588684,2.384110,1,1,1,2,NaN,12.305115,NaN
2,2025-01-01 02:00:00,26.0,83.0,12.506287,9.928528,1010.119385,2.863358,-0.085052,49.688721,49.688721,...,0.000000,84.298172,2.864620,2,1,1,2,NaN,12.614502,NaN
3,2025-01-01 03:00:00,26.0,83.0,12.865356,9.241760,1010.722656,2.798767,-0.625854,42.849731,42.849731,...,0.000000,78.629288,2.867890,3,1,1,2,NaN,12.506287,NaN
4,2025-01-01 04:00:00,26.0,83.0,12.227509,7.806671,1011.052490,3.277405,-0.995514,38.131714,38.131714,...,0.000000,74.398216,3.425264,4,1,1,2,NaN,12.865356,NaN


## Rain Classification Target

In [12]:
weather_df["rain_tomorrow"] = (
    weather_df["precipitation"]
    .shift(-24) > 0
).astype(int)

weather_df['rain_tomorrow'].value_counts()

rain_tomorrow
0    6007
1    2753
Name: count, dtype: int64

## Drop Irrelevant Columns

In [35]:
weather_df = weather_df.drop(columns=['latitude', 'longitude', 'number', 'expver', 'wind_u', 'wind_v', 'd2m'])

## Remove Missing Values

In [36]:
weather_df = weather_df.dropna()

print(weather_df.shape)
weather_df.head()

(8736, 18)


,valid_time,temperature,surface_pressure,total_cloud_cover,low_cloud_cover,medium_cloud_cover,high_cloud_cover,precipitation,humidity,wind_speed,hour,day,month,day_of_week,temp_rolling_6,temp_lag_1,temp_lag_24,rain_tomorrow
24,2025-01-02 00:00:00,10.243958,1007.981079,0.000000,0.000000,0.0,0.000000,0.0,91.275276,1.957509,0,2,1,3,9.957026,9.754242,12.305115,0
25,2025-01-02 01:00:00,10.034027,1008.836182,0.000000,0.000000,0.0,0.000000,0.0,91.548195,1.777704,1,2,1,3,9.885452,10.243958,12.614502,0
26,2025-01-02 02:00:00,10.117249,1009.328125,0.015259,0.015259,0.0,0.000000,0.0,89.642296,2.108368,2,2,1,3,9.903219,10.034027,12.506287,0
27,2025-01-02 03:00:00,10.726746,1010.089478,0.000000,0.000000,0.0,0.000000,0.0,86.600449,1.495689,3,2,1,3,10.073166,10.117249,12.865356,0
28,2025-01-02 04:00:00,11.140686,1010.746155,0.082397,0.048828,0.0,0.033569,0.0,85.779327,1.550697,4,2,1,3,10.336151,10.726746,12.227509,0


## Checking DataType of Features

In [37]:
weather_df.dtypes

valid_time            datetime64[ns]
temperature                  float32
surface_pressure             float32
total_cloud_cover            float32
low_cloud_cover              float32
medium_cloud_cover           float32
high_cloud_cover             float32
precipitation                float32
humidity                     float32
wind_speed                   float32
hour                           int32
day                            int32
month                          int32
day_of_week                    int32
temp_rolling_6               float64
temp_lag_1                   float32
temp_lag_24                  float32
rain_tomorrow                  int64
dtype: object

## Save Final Dataset

In [38]:
# =========================
# OUTPUT CONFIGURATION
# =========================

OUTPUT_DIR = DATASET_DIR / "processed"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "azamgarh_weather_final1.csv"
)


# =========================
# SAVE DATASET
# =========================

weather_df.to_csv(
    OUTPUT_FILE,
    index=False
)


# =========================
# CONFIRMATION
# =========================

print("\nDataset saved successfully.")

print(f"\nLocation: {OUTPUT_FILE}")

print(f"\nTotal Rows: {weather_df.shape[0]}")

print(f"Total Columns: {weather_df.shape[1]}")


Dataset saved successfully.

Location: c:\Users\DELL\OneDrive\Desktop\weather-forecasting-system\data\processed\azamgarh_weather_final1.csv

Total Rows: 8736
Total Columns: 18
